In [ ]:
# Define the list of “politics” topic IDs you care about and other arguments
politics_topics = [0, 2, 4, 5, 6, 7, 8, 13, 16, 18, 30, 36, 43, 44, 57]
economy_topics = [1, 3, 41, 45, 46, 54]
crypto_topics = [14, 17, 21, 22, 24, 40, 42, 50, 61, 63, 64]
not_categorized_topics = [9, 10, 11, 12, 15, 19, 20, 23, 25, 26, 27, 28, 29, 31, 32, 33,
                  34, 35, 37, 38, 39, 47, 48, 49, 51, 52, 53, 55, 56, 58, 59, 60, 62, 65, 66]

In [ ]:
# Parameters
input = "0"  # default; papermill will override with -p input <value>

# papermill visualize_gridSearch_results.ipynb out.ipynb -k bertopic_env -p input 0

# --- Parameters / CLI parsing (works in notebook and CLI) ---
# papermill visualize_gridSearch_results.ipynb out.ipynb -k bertopic_env -p input 0
# or:
# jupyter nbconvert --to notebook --execute visualize_gridSearch_results.ipynb --ExecutePreprocessor.timeout=-1 --output out.ipynb

import os
import argparse
from pathlib import Path
import pandas as pd
from bertopic import BERTopic
import plotly.io as pio
from pathlib import Path
from bertopic import BERTopic
import pandas as pd
import plotly.io as pio
from bertopic import BERTopic
from pathlib import Path
import pandas as pd
import plotly.express as px
import pandas as pd
import numpy as np

# Headless-safe Plotly renderer
pio.renderers.default = "notebook_connected" if (os.environ.get("DISPLAY") or os.environ.get("MPLBACKEND")) else "png"

# Parse CLI args
parser = argparse.ArgumentParser(description="BERTopic model loader + charts")
parser.add_argument("--input", type=str, default="0", help="level depth (e.g., 0, 1, 2, ...)")
args, _ = parser.parse_known_args()

# Allow papermill injected variable named 'input'
if "input" in globals() and isinstance(globals()["input"], (str, int)):
    level_depth = str(globals()["input"])
else:
    level_depth = args.input

print(f"[INFO] level_depth={level_depth}")

In [ ]:
# DF_SAMPLED
# here we are working only with the SEEDS

#level_depth parameyer
level_depth = 0

#paths
path_df_sampled = f"../results/levels/level_{level_depth}/grid_search/df_sampled_level_{level_depth}.csv"
best_dir = f"../results/levels/level_{level_depth}/best"
best_model_path = os.path.join(best_dir, "best_model.pkl")
df_spam_messages_path = f"../results/levels/level_{level_depth}/preProcessing/preprocessed_spam_messages_level_{level_depth}.tsv.gz"
path_summary_filtered = f"../results/levels/level_{level_depth}/percentage_of_politics_msgs/summary_filtered.csv.gz"


# Load model and dataframe
topic_model = BERTopic.load(best_model_path)
df_sampled = pd.read_csv(path_df_sampled)


# Attach the assigned topic to each message
df_sampled["topic"] = topic_model.topics_

#DF_SPAM_MESSAGES
df_spam_messages = pd.read_csv(df_spam_messages_path, sep='\t', compression='gzip')

"""
df_spam_messages
| channel\_id | spam_msg                  | count_spam |
| ----------- | ------------------------- | -----      |
| 1001        | join our crypto group now | 2          |
| 1001        | hello everyone            | 1          |
| 1002        | vote for freedom          | 3          |
| 1002        | hello                     | 1          |
"""

#DF_TOTAL_SPAM
df_spam_messages["count_spam"] = df_spam_messages["count_spam"].clip(lower=0)  
df_total_spam = (
    df_spam_messages
    .groupby("channel_id")["count_spam"]
    .sum()
    .reset_index(name="spam_msgs")
)

In [ ]:

# Group by channel_id and compute:
#    - total_msgs_without_spam: total number of messages in that channel
#    - pol_msgs:   how many of those messages belong to the politics_topics
#    - merge with spam
# grouping

#removing spam messages from df_sampled
spam_texts = set(df_spam_messages["text_preprocessed"])
df_sampled_without_spam = df_sampled[~df_sampled["text_preprocessed"].isin(spam_texts)].copy()
summary = (
    df_sampled_without_spam
    .groupby("channel_id")["topic"]
    .agg(
        total_msgs_without_spam   = "count",
        pol_msgs     = lambda s: s.isin(politics_topics).sum(),
        economy_msgs = lambda s: s.isin(economy_topics).sum(),
        crypto_msgs  = lambda s: s.isin(crypto_topics).sum(),
        not_categorized_msgs   = lambda s: s.isin(not_categorized_topics).sum(),
    )
    .reset_index()
)

#merging
summary = summary.merge(df_total_spam, on="channel_id", how="left" )
summary["spam_msgs"] = summary["spam_msgs"].fillna(0).astype(int)

#adding spam count 
summary["outliers_msgs"] = summary["total_msgs_without_spam"] - summary["pol_msgs"] - summary["economy_msgs"] - summary["crypto_msgs"] - summary["not_categorized_msgs"] 
summary["total_msgs_with_spam"] = summary["total_msgs_without_spam"] + summary["spam_msgs"] 

# calculate all the ratio
summary = summary.assign(
    pol_ratio_without_spam     = lambda d: d["pol_msgs"]   / d["total_msgs_without_spam"] * 100,
    economy_ratio_without_spam = lambda d: d["economy_msgs"]/ d["total_msgs_without_spam"] * 100,
    crypto_ratio_without_spam  = lambda d: d["crypto_msgs"]/ d["total_msgs_without_spam"] * 100,
    not_categorized_ratio_without_spam   = lambda d: d["not_categorized_msgs"] / d["total_msgs_without_spam"] * 100,
    spam_ratio    = lambda d: d["spam_msgs"] / d["total_msgs_with_spam"] * 100
)

# Filter: only channels with more than 100 total messages
summary_filtered = summary[summary["total_msgs_without_spam"] > 100]

# Sort by political message ratio
summary_sorted = summary_filtered.sort_values(by="pol_ratio_without_spam", ascending=False)

# Filter: only rows with pol_ratio_without_spam > 0.2
summary_filtered = summary_sorted[summary_sorted["pol_ratio_without_spam"] > 0.2]
summary_filtered = summary_filtered.reset_index(drop=True)


# Define the bins and the truncated % labels
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]
labels = ["0%", "10%", "20%", "30%", "40%", "50%", "60%", "70%", "80%", "90%"]

# Assign each channel to a bin based on political ratio
summary_filtered["pol_ratio_without_spam_range"] = pd.cut(summary_filtered["pol_ratio_without_spam"], bins=bins, labels=labels, right=False)

#save summary filtered  
summary_filtered.to_csv(path_summary_filtered, sep=",", index=False, compression="gzip")


'''
   channel_id  total_msgs_without_spam  pol_msgs  economy_msgs  crypto_msgs  \
0  1840578235                      540       310            45           12   
1  2036850729                      260       120            12            8   
2  1306559115                     1240       780            40           15   
3  1749991917                      180        55            18            9   
4  1581117699                      415        92            33           60   

   not_categorized_msgs  spam_msgs  outliers_msgs  total_msgs_with_spam  \
0                   150         60             23                   600   
1                   110         40             10                   300   
2                   360        155             45                  1395   
3                    85         20             13                   200   
4                   205         35             25                   450   

   pol_ratio_without_spam  economy_ratio_without_spam  \
0                   57.41                        8.33   
1                   46.15                        4.62   
2                   62.90                        3.23   
3                   30.56                       10.00   
4                   22.17                        7.95   

   crypto_ratio_without_spam  not_categorized_ratio_without_spam  spam_ratio  \
0                       2.22                                27.78      10.00   
1                       3.08                                42.31      13.33   
2                       1.21                                29.03      11.11   
3                       5.00                                47.22      10.00   
4                      14.46                                49.40       7.78   

  pol_ratio_without_spam_range
0                         50%
1                         40%
2                         60%
3                         30%
4                         20%
'''